In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Определение переменных
data_source = './data/'

### Чтение данных

In [ ]:
df_clean = pd.read_csv(data_source + 'gpt_etalon_df_labeled.csv')
display(df_clean.info())
display(df_clean.head())

## Обучение моделей


### Препроцессинг набора данных

In [ ]:
# Импортируем библиотеки
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report
from joblib import parallel_backend # Импортируем библиотеку для параллельных расчетов

metrics_dict = {} # В этот словарь будем сохранять все метрики


In [ ]:
def bp_gridsearch(X_train: pd.DataFrame, y_train: pd.DataFrame,
                  model, params: dict, scoring: str, cv: int,
                  print_params=False, jobs=5):
    """Перебор гиперпараметров по сетке с ипользованием параллелизма

    Parameters
    ----------
        X_train : DataFrame
            объекты обучающей выборки
        y_train : DataFrame
            значения целевой переменной обучающей выборки
        model : any
            модель, для которой производится подбор гиперпараметров
        params : dict
            словарь с названиями гиперпараметрамов и всеми наборами значений
        scoring : str
            функция, которая будет минимизироваться в ходе кросс-валидации
        cv : int
            количество частей, на которые будет поделена выборка
        print_params : bool
            вывод в консоль оптимальных гиперпараметров
        jobs : int 
            количество параллельных потоков

    Returns
    -------
        best_params: dict
            наилучшие гиперпараметры
    """
    searcher = GridSearchCV(
        estimator=model,
        param_grid=params,
        scoring=scoring,
        cv=cv,
        verbose=0,
    )
    with parallel_backend('threading', n_jobs=jobs):
        searcher.fit(X_train, y_train)
    best_params = searcher.best_params_
    if print_params:
        print(f'Оптимальные гиперпараметры:\n{'=' * 27}\n'
            f'{pd.DataFrame.from_dict(best_params,
                                      orient='index',
                                      columns=['значение'])}\n')
    return best_params


In [ ]:
def plot_imp(features: list, importance: np.ndarray, model_name=''):
    """Построение диаграммы важности признаков

    Parameters
    ----------
       features : list
         Названия признаков
       importance : np.ndarray
         Важность признаков из модели
       model_name : str 
         Наименование модели
    """
    # Сортируем массив со значениям и важности признаков
    order = np.argsort(importance)
    # Создаем датафрейм с парами название признака - важность в порядке убывания
    feat_data = pd.DataFrame({
    'features': features[order][::-1],
    'importance': importance[order][::-1]
    })
    # Округлим значение важности до 3 знаков после точки
    feat_data['importance'] = round(feat_data['importance'], 3)
    # Отображение диаграммы
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(feat_data, x='importance', y='features')
    ax.set(title=f'Важность признаков в модели {model_name}',
           xlabel='Важность', ylabel='Признаки')
    ax.bar_label(ax.containers[0], fontsize=8);


In [ ]:
# Стандартизация числовых признаков в тренировочном и валидационном наборе
from sklearn.preprocessing import StandardScaler
x_scaler = StandardScaler()
df_clean[num_cols_list] = x_scaler.fit_transform(df_clean[num_cols_list])
# Разделение на массив признаков и целевую переменную
X = df_clean.drop(columns=['class'])
y = df_clean['class']

# Разделим набор данных на обучающую и тестовую
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Посмотрим на баланс классов
val_count_init = pd.Series(df_clean['class']).value_counts().to_dict()
val_count_train = y_train.value_counts().to_dict()
val_count_test = y_test.value_counts().to_dict()

get_relation = lambda x: round(x[1] / x[0], 2)

print(
    """
    Соотношение классов в исходной выборке:   {}\t(класс 1/класс 0 = {})
    Соотношение классов в обучающей выборке:  {}\t(класс 1/класс 0 = {})
    Соотношение классов в тестовой выборке:   {}\t(класс 1/класс 0 = {})
    """.format(
        val_count_init, get_relation(val_count_init),
        val_count_train, get_relation(val_count_train),
        val_count_test, get_relation(val_count_test),
    )
)


### Логистическая регрессия

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(random_state=42)
y_pred = lr.fit(X_train, y_train).predict(X_test)
print(f'Метрики модели LogisticRegression с гиперпараметрами по умолчанию:\n{'=' * 70}\n'
      f'{classification_report(y_test, y_pred)}')
# Сохраним метрики для сравнения
metrics_dict['LogisticRegression'] = classification_report(y_test, y_pred,
                                            output_dict=True)['macro avg']
lr_params = {
    "penalty": [None, 'l2'],
    # "solver": ['lbfgs', 'liblinear'],
    # "l1_ratio": [0.5, 0.25],
    "max_iter": [50, 75, 100],
    "random_state": [42]
}
lr_bp = bp_gridsearch(X_train, y_train,
                   LogisticRegression(),
                   lr_params, 'recall', 3, True, 7)
best_lr = LogisticRegression(**lr_bp)
y_pred = best_lr.fit(X_train, y_train).predict(X_test)
print(f'Метрики модели LogisticRegression с оптимальными гиперпараметрами:\n{'=' * 70}\n'
      f'{classification_report(y_test, y_pred)}')
# Сохраним метрики для сравнения
metrics_dict['LogisticRegression_opt'] = classification_report(y_test, y_pred,
                                            output_dict=True)['macro avg']
plot_imp(df_clean.columns, np.abs(lr.coef_[0]), 'LogisticRegression')


### Случайный лес


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42)
y_pred = rf.fit(X_train, y_train).predict(X_test)
print(f'Метрики модели RandomForestClassifier с гиперпараметрами по умолчанию:\n{'=' * 70}\n'
      f'{classification_report(y_test, y_pred)}')
# Сохраним метрики для сравнения
metrics_dict['RandomForest'] = classification_report(y_test, y_pred,
                                            output_dict=True)['macro avg']
plot_imp(df_clean.columns, np.abs(rf.feature_importances_), 'RandomForestClassifier')


In [ ]:

from catboost import CatBoostClassifier
catgb = CatBoostClassifier(random_state=42, verbose=False)
y_pred = catgb.fit(X_train, y_train).predict(X_test)
print(f'Метрики модели CatBoostClassifier с гиперпараметрами по умолчанию:\n{'=' * 66}\n'
      f'{classification_report(y_test, y_pred)}')
# Сохраним метрики для сравнения
metrics_dict['CatBoost'] = classification_report(y_test, y_pred,
                                            output_dict=True)['macro avg']
plot_imp(df_clean.columns, np.abs(catgb.feature_importances_), 'CatBoostClassifier')


In [ ]:
##### Общая производительность
CatBoostClassifier показывает лучшие результаты по сравнению с LogisticRegression:
- Accuracy: 0.90 против 0.88
- Macro avg метрики: 0.90 против 0.88
- Weighted avg метрики: 0.90 против 0.88
##### Анализ по классам
**Для класса 0:**
- CatBoost: precision=0.91, recall=0.92, f1=0.91
- LogisticRegression: precision=0.89, recall=0.90, f1=0.90

**Для класса 1:**
- CatBoost: precision=0.90, recall=0.88, f1=0.89
- LogisticRegression: precision=0.88, recall=0.86, f1=0.87
##### Ключевые наблюдения
CatBoostClassifier демонстрирует более высокую полноту (recall) для класса 1 (88% против 86%), что означает лучшее обнаружение положительных случаев.

Обе модели показывают схожие результаты, разница составляет около 2% по основным метрикам.

CatBoostClassifier превосходит LogisticRegression по большинству метрик, особенно по recall для класса 1. Однако разница в производительности невелика.
##### Вывод:
Принимая во внимание то, что в решении медицинских задач более важно обнаружение положительных результатов (высокий recall), CatBoost, как и RandomForest больше подходят для решения этой задачи. Однако, стоит отметить, что обучение CatBoost происходит быстрее.

#### Значимость признаков
Три наиболее важных признака:
1. maximum_heart_rate_achved: максимальная частота сердечных сокращений в минуту
2. thal_6: 6 = фиксированный дефект
3. age: возраст пациента


В данной модели важность признаков распределилась иначе, однако результаты исследования миокарда с таллием по прежнему входят в три наиболее важных признака. В общем, такое распределение соответствует клинической картине.


In [ ]:
### Нейронная сеть
import tensorflow as tf
import keras
from keras.layers import Input, Dense
from keras.models import Sequential
from keras.backend import clear_session
from keras.callbacks import ModelCheckpoint
clear_session()
model = Sequential()
model.add(Input(shape=(X_train.shape[1],))) # входной слой
model.add(Dense(64, activation='sigmoid',)) # первый скрытый слой
model.add(Dense(32, activation='relu',)) # второй скрытый слой
# model.add(Dense(16, activation='relu',)) # третий скрытый слой
model.add(Dense(1, activation='sigmoid',)) # выходной слой
model.summary()
model.compile(
    loss='binary_crossentropy',  # минимизируем кросс-энтропию
    optimizer='adam',
    metrics=['recall', 'accuracy']  # приоритет recall
)

callbacks = [
    ModelCheckpoint("checkpoints/epoch_{epoch}.keras")
]
model_history = model.fit(
                    X_train,
                    y_train,
                    batch_size=64,  # 64 объекта для подсчета градиента на каждом шаге
                    epochs=10,  # 10 проходов по датасету
                    callbacks=[callbacks],
                    validation_data=(X_test, y_test)
                    )
# Характеристики процесса обучения.
loss = model_history.history["loss"]
val_loss = model_history.history["val_loss"]
epochs = list(map(lambda epoch: epoch + 1, model_history.epoch))  # Преобразуем нумерацию эпох (с 0 -> с 1).

plt.figure()
plt.plot(epochs, loss, "b", label="Обучение")
plt.plot(epochs, val_loss, "g", label="Валидация")
plt.title("Loss при обучении и валидации")
plt.xlabel("Эпоха")
plt.xticks(range(1, 10 + 1))
plt.ylim([0.2, 0.35])
plt.legend()
plt.show()
# Выбираем эпоху с лучшими показателями
model = keras.saving.load_model('checkpoints/epoch_10.keras')
y_pred = model.predict(X_test)
# y_pred = tf.squeeze(y_pred)
y_pred = np.array([1 if x >= 0.5 else 0 for x in y_pred])
print(f'Метрики модели NeuroNetwork:\n{'=' * 70}\n'
      f'{classification_report(y_test, y_pred)}')
# Сохраним метрики для сравнения
metrics_dict['NeuralNetwork'] = classification_report(y_test, y_pred,
                                            output_dict=True)['macro avg']
Спроектирована и обучена полносвязная нейронная сеть.

##### Общая производительность
Нейронная показывает лучшие результаты по сравнению с LogisticRegression, однако схожие с CatBoostClassifier:
- Accuracy: 0.89 против 0.88
- Macro avg метрики: 0.89 против 0.88
- Weighted avg метрики: 0.90 против 0.88
##### Анализ по классам
**Для класса 0:**
- Нейронная сеть: precision=0.92, recall=0.88, f1=0.9
- LogisticRegression: precision=0.89, recall=0.90, f1=0.90

**Для класса 1:**
- CatBoost: precision=0.86, recall=0.91, f1=0.88
- LogisticRegression: precision=0.88, recall=0.86, f1=0.87
##### Ключевые наблюдения
Нейронная сеть демонстрирует более высокую полноту (recall) для класса 1 (91% против 86%), что означает лучшее обнаружение положительных случаев.

Нейронная сеть превосходит LogisticRegression по большинству метрик, особенно по recall для класса 1.
##### Вывод:
Принимая во внимание то, что в решении медицинских задач более важно обнаружение положительных результатов (высокий recall), нейронная сеть больше подходят для решения этой задачи. Однако, стоит отметить, что обучение нейронной сети происходит медленнее и требует больших ресурсов.
## Сравнение моделей
Для сравнения всех примененных моделей сравним их метрики: Accuracy и усредненные precision, recall, f1-score.

Т.к. классы в выборках сбалансированы будем ориентироваться на Macro average
pd.DataFrame.from_dict(metrics_dict, orient='index').sort_values(by=['recall', 'f1-score'], ascending=False).round(3)
Наилучшие метрики демонстрирует CatBoostClassifier.

У неронной сети и случайного леса метрики одинаковые. Метрики логистической регрессии ниже на 1%
У моделей, вошедших в топ-3 оценим разницу метрик в тренировочных и тестовых наборах для оценки переобучения моделей.
# проверим топ моделей на переобучение
overf_met_dict = {}

def overfit_metrics(model, model_label):
    """Расчет разницы метрик тестового и тренировочного 
    набора для оценки переобучения моделей
    """
    # Сделаем предсказание на тестовом и тренировочном наборе
    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)

    y_pred_test = np.array([1 if x >= 0.5 else 0 for x in y_pred_test])
    y_pred_train = np.array([1 if x >= 0.5 else 0 for x in y_pred_train])

    # Расчитаем тестовые и тренировочные метрики
    test_metrics = classification_report(y_pred_test, y_test, output_dict=True)
    train_metrics = classification_report(y_pred_train, y_train, output_dict=True)
    # Вычислим разницу в метриках между тренировочным и тестовым набором
    overf_met_dict[model_label] = {
        'accuracy_dif': train_metrics['accuracy'] - test_metrics['accuracy'],
        'recall_dif': train_metrics['macro avg']['recall'] - test_metrics['macro avg']['recall'],
        'f1-score_dif': train_metrics['macro avg']['f1-score'] - test_metrics['macro avg']['f1-score']
        }
    
best_model_dict = {'RandomForest': rf,
                #    'lr': lr,
                #    'best_lr': best_lr,
                   'CatBoost': catgb,
                   'NeuralNetwork': model, 
                   }

for l, m in best_model_dict.items():
    overfit_metrics(m, l)
pd.DataFrame.from_dict(overf_met_dict, orient='index').sort_values(by=['recall_dif', 'f1-score_dif']).round(3)
У всех моделей разница в метриках небольшая. Наименьшая у нейронной сети - 0.1%. У лидера, CatBoostClassifier тенденция к переобучению выше - 1%, но эту разницу нельзя назвать большой.

Самая большая разница между метриками у RandomForestClassifire с параметрами по умолчанию - 10%. 

### Выбор лучших моделей

Примая во внимание разницу в метриках в десятые доли процентов и признаки переобуения по основным метрикам, лучшей моделью показала себя **полносвязная нейронная сеть**. Однако, стоит отметить, что при обучении эта модель демонстрировала наибольшее время и более высокую требовательность к ресурсам. Это стоит учитывать, если в дальнейшем будет необходимо проводить дообучение на новых данных.

Вторая модель, которую можно рекомендовать к использованию - **CatBoostClassifier**. Данная модель высокую точность, как неронная сеть, однако время обучения значительно выше.


# Сохраняем инференсы лучших моделей

import joblib
joblib.dump(catgb, 'inference/catboost.pkl')

model.save("inference/model_classification.keras")
## Общий вывод
### В ходе работы было проведено:
1. Загрузка данных, разведочный анализ и первичная очистка. Были удалены неинформативные признаки и выбросы.
2. Проведена проверка на мультиколлинеарность с помощью матрицы корреляций. Чистая мультиколлинеарность и сильные корреляции выявлены не были.
3. Была проведена стандартизация данных.
4. Был выполнен подбор лучших моделей из 4-х: LogisticRegression, RandomForestClassifire, CatBoostClassifire, нейронная сеть. Можно рекомендовать к применению **полносвязная нейронная сеть** и **CatBoostClassifire**.
5. Определены наиболее значимые признаки, влияющие на развитие сердечно-сосудистых заболеваний, это - фиксированный дефект миокарда при исследовании таллием, высокая максимальная частота сердечных сокращений, 2 или 1 видимых крупных сосуда при ангиографии.
### Рекомендации для дальнейшего повышения качества моделей:
1. Создание новых признаков на основе существующих и обучение моделей с их использованием.
2. Кодирование категориальных признаков и обучение, в том числе, с применением алгоритмов понижения размерности.
3. Обучение моделей, которые спроектированы для работы с категориальными признаками, на наборе данных без стандартизации.
